In [2]:
# pip install kagglehub
# pip install altair
# pip install mpld3

In [ ]:
import altair as alt
import mpld3

# Allow larger datasets in charts
alt.data_transformers.enable('default', max_rows=None)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import altair as alt

# Download latest version
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")
print("Path to dataset files:", path)

# List all files in the downloaded directory
print("\nAvailable files:")
for file in os.listdir(path):
    print(f"  - {file}")

# Load the movies dataset
movies_df = pd.read_csv(os.path.join(path, 'tmdb_5000_movies.csv'))
print(f"\nMovies dataset shape: {movies_df.shape}")
print(f"Movies columns: {list(movies_df.columns)}")

# Load the credits dataset
credits_df = pd.read_csv(os.path.join(path, 'tmdb_5000_credits.csv'))
print(f"\nCredits dataset shape: {credits_df.shape}")
print(f"Credits columns: {list(credits_df.columns)}")

# Display first few rows of movies dataset
print("\nFirst 5 rows of movies dataset:")
print(movies_df.head())

# Merge the two datasets on 'title'
merged_df = pd.merge(movies_df, credits_df, on=['title'], how='left')
print(f"\nMerged dataset shape: {merged_df.shape}")

In [ ]:
# 1. Clean and parse the release_date column
merged_df['release_date'] = pd.to_datetime(merged_df['release_date'], errors='coerce')
merged_df = merged_df.dropna(subset=['release_date'])

# 2. Extract month from release date
merged_df['release_month'] = merged_df['release_date'].dt.month
merged_df['release_year'] = merged_df['release_date'].dt.year

# 3. Clean revenue column - remove rows with 0 or null revenue
merged_df = merged_df[merged_df['revenue'] > 0]
print(f"\nDataset after removing zero revenue: {merged_df.shape}")

# 4. Parse genres column (it's in JSON format)
def parse_genres(genres_str):
    try:
        genres_list = json.loads(genres_str.replace("'", '"'))
        return [genre['name'] for genre in genres_list]
    except:
        return []

merged_df['genres_list'] = merged_df['genres'].apply(parse_genres)

# 5. Explode genres so each movie-genre combination gets its own row
exploded_df = merged_df.explode('genres_list')
exploded_df = exploded_df[exploded_df['genres_list'].notna()]
exploded_df = exploded_df[exploded_df['genres_list'] != '']

print(f"\nUnique genres found: {exploded_df['genres_list'].nunique()}")
print(f"Genres: {sorted(exploded_df['genres_list'].unique())}")

In [ ]:
roi_df = exploded_df.copy()
roi_df = roi_df[(roi_df['budget'] > 0) & (roi_df['revenue'] > 0)]
roi_df['roi'] = (roi_df['revenue'] / roi_df['budget'] * 100)
roi_df = roi_df[roi_df['roi'] < 3000]

top_genres = roi_df["genres_list"].value_counts().head(6).index
roi_top = roi_df[roi_df["genres_list"].isin(top_genres)].copy()
roi_top['release_year'] = roi_top['release_year'].astype(int)
roi_top = roi_top[roi_top['release_year'] >= 2005].copy()


year_options = sorted(roi_top['release_year'].unique())
default_year = int(min(year_options))

year_param = alt.param(
    name='Year',
    value=default_year,
    bind=alt.binding_range(
        min=int(min(year_options)),
        max=int(max(year_options)),
        step=1
))

genre_selection = alt.selection_multi(
    fields=['genres_list'],
    bind='legend'    
)

base = (
    alt.Chart(roi_top)
    .mark_point(filled=True, size=28, opacity=0.7)
    .encode(
        x=alt.X(
            'budget:Q',
            title='Budget',
        ),
        y=alt.Y(
            'roi:Q',
            title='ROI (%)',
        ),
        color=alt.condition(
            genre_selection,
            alt.Color(
                'genres_list:N',
                title='Genre',
                scale=alt.Scale(scheme='tableau10')
            ),
            alt.value('lightgrey')
        ),
        tooltip=[
            alt.Tooltip('title:N', title='Movie'),
            alt.Tooltip('genres_list:N', title='Genre'),
            alt.Tooltip('release_year:Q', title='Year'),
            alt.Tooltip('budget:Q', title='Budget', format='$,.0f'),
            alt.Tooltip('revenue:Q', title='Revenue', format='$,.0f'),
            alt.Tooltip('roi:Q', title='ROI', format='.2f')
        ]
    )
    .add_params(year_param, genre_selection)
    .transform_filter(alt.datum.release_year <= year_param)
    .properties(
        width=700,
        height=450,
        title={
            'text': 'ROI vs Budget for Top Genres',
            'subtitle': 'Filter by year with the slider and by genre using the legend'
        }
    )
    .interactive()
)

base.save('viz_exports/roi_budget_chart.html')

base

In [ ]:
start_year_param = alt.param(
    name='start_year',
    value=int(roi_top['release_year'].min()),
    bind=alt.binding_range(
        min=int(roi_top['release_year'].min()),
        max=int(roi_top['release_year'].max()),
        step=1,
        name='Start Year: '
    )
)

end_year_param = alt.param(
    name='end_year',
    value=int(roi_top['release_year'].max()),
    bind=alt.binding_range(
        min=int(roi_top['release_year'].min()),
        max=int(roi_top['release_year'].max()),
        step=1,
        name='End Year: '
    )
)

chart = alt.Chart(roi_top).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('vote_average:Q', title='Rating', scale=alt.Scale(domain=[0, 10])),
    y=alt.Y('roi:Q', title='ROI (%)'),
    color=alt.Color('release_year:Q', 
                    scale=alt.Scale(scheme='tableau10'),
                    legend=alt.Legend(format='d', title='Year')),
    tooltip=['title:N', 'release_year:Q', 'vote_average:Q', 'roi:Q', 'genres_list:N']
).add_params(
    start_year_param,
    end_year_param
).transform_filter(
    (alt.datum.release_year >= start_year_param) &
    (alt.datum.release_year <= end_year_param)
).properties(
    width=600,
    height=400,
    title='Rating vs ROI Over Time'
)

regression_line = alt.Chart(roi_top).add_params(
    start_year_param,
    end_year_param
).transform_filter(
    (alt.datum.release_year >= start_year_param) &
    (alt.datum.release_year <= end_year_param)
).transform_regression(
    'vote_average',
    'roi'
).mark_line(color='black', strokeWidth=2).encode(
    x=alt.X('vote_average:Q', scale=alt.Scale(domain=[0, 10])),
    y=alt.Y('roi:Q')
)

chart = chart + regression_line
chart.save('viz_exports/roi_rating_chart.html')
chart

In [ ]:
import matplotlib.colors as mcolors

tab10 = plt.cm.tab10.colors
revenue_cmap = mcolors.LinearSegmentedColormap.from_list(
    'revenue', 
    [tab10[0], "#C7E9B4", "#FFEB99", "#FDD49E", tab10[2], tab10[1]])

# Calculate average revenue by genre and month
genre_month_revenue = exploded_df.groupby(['genres_list', 'release_month'])['revenue'].agg([
    'mean',
    'count'
]).reset_index()

# Filter out genre-month combinations with too few movies (less than 5)
genre_month_revenue = genre_month_revenue[genre_month_revenue['count'] >= 5]

heatmap_data = genre_month_revenue.pivot(
    index='genres_list', 
    columns='release_month', 
    values='mean'
)

heatmap_data = heatmap_data.fillna(0)

# Convert revenue to millions for better readability
heatmap_data = heatmap_data / 1_000_000


plt.figure(figsize=(14, 10))

# Create month labels
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


# Create heatmap with annotations
ax = sns.heatmap(
    heatmap_data,
    cmap=revenue_cmap,  
    annot=True,      
    fmt='.1f',       
    cbar_kws={'label': 'Average Revenue (Million USD)'},
    xticklabels=month_labels,
    yticklabels=True,
    linewidths=0.5,
    linecolor='gray',
    alpha=0.9,
    annot_kws={'size': 8, 'weight': 'bold'} 
)

plt.title('Movie Genre Success by Release Month\n(Based on Average Revenue)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Release Month', fontsize=12, fontweight='bold')
plt.ylabel('Genre', fontsize=12, fontweight='bold')

plt.yticks(rotation=0)
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig("viz_exports/genre_month_heatmap.png", dpi=150, bbox_inches="tight")

plt.show()

In [ ]:
plot_df = exploded_df[
    (exploded_df['budget'] > 0) & (exploded_df['budget'] <= 2e8)
].copy()

top_genres = plot_df['genres_list'].value_counts().head(6).index.tolist()
plot_df_filtered = plot_df[plot_df['genres_list'].isin(top_genres)].copy()

TAB10 = [
    "#1f77b4",  
    "#ff7f0e",  
    "#2ca02c",  
    "#d62728",  
    "#9467bd",  
    "#8c564b",  
    "#e377c2",  
    "#7f7f7f",  
    "#bcbd22",  
    "#17becf",  
]

base_color_scale = alt.Scale(
    domain=top_genres,
    range=TAB10[:len(top_genres)]
)

brush = alt.selection_interval(encodings=['x', 'y'])

scatter = alt.Chart(plot_df_filtered).mark_circle(size=60, clip=True).encode(
    x=alt.X(
        'budget:Q',
        title='Budget (log scale)',
        scale=alt.Scale(type='log', domain=[1e5, 2e8]),
        axis=alt.Axis(format='$,.0s', tickCount=6)
    ),
    y=alt.Y(
        'vote_average:Q',
        title='Average Rating',
        scale=alt.Scale(domain=[4, 9])
    ),
    color=alt.Color(
        'genres_list:N',
        title='Genre',
        scale=base_color_scale
    ),
    opacity=alt.condition(brush, alt.value(0.8), alt.value(0.15)),
    tooltip=[
        alt.Tooltip('title:N', title='Movie'),
        alt.Tooltip('genres_list:N', title='Genre'),
        alt.Tooltip('budget:Q', format='$,.0f', title='Budget'),
        alt.Tooltip('vote_average:Q', format='.1f', title='Average Rating'),
        alt.Tooltip('vote_count:Q', format=',', title='Vote Count')
    ],
)

regression_lines = (
    alt.Chart(plot_df_filtered)
    .transform_regression(
        'budget',
        'vote_average',
        groupby=['genres_list'],
    )
    .mark_line(strokeWidth=2, opacity=0.8, clip=True)
    .encode(
        x=alt.X(
            'budget:Q',
            scale=alt.Scale(type='log', domain=[1e5, 2e8]),
            axis=alt.Axis(format='$,.0s', tickCount=6)
        ),
        y=alt.Y(
            'vote_average:Q',
            scale=alt.Scale(domain=[4, 9])
        ),
        color=alt.Color(
            'genres_list:N',
            scale=base_color_scale,
            title='Genre',
        ),
    )
)

scatter_chart_with_regressions = (
    alt.layer(scatter, regression_lines)
    .properties(
        width=700,
        height=450,
        title='Movie Budget vs Average Rating (log budget)',
    )
    .add_params(brush)
)

genre_bar = (
    alt.Chart(plot_df_filtered)
    .mark_bar(clip=True)
    .encode(
        x=alt.X('genres_list:N', title='Genre'),
        y=alt.Y('count():Q', title='Number of Movies'),
        color=alt.Color('genres_list:N', scale=base_color_scale, legend=alt.Legend(title='Genre')),
        tooltip=[
            alt.Tooltip('genres_list:N', title='Genre'),
            alt.Tooltip('count():Q', title='Number of Movies'),
        ],
    )
    .transform_filter(brush)
    .properties(
        width=700,
        height=200,
        title='Genre distribution of selected movies',
    )
)

chart = alt.vconcat(
    scatter_chart_with_regressions,
    genre_bar,
).resolve_scale(color='shared')

chart.save('viz_exports/budget_rating_chart.html')
chart


Takeaway: This visualization shows that higher movie budgets do not guarantee better audience ratings. Across different budgets, most films cluster between 4 and 8 on the rating scale, and the regression line stays mostly flat. Some low- and mid-budget films achieve strong ratings, while expensive productions still get average or slightly above-average scores. Ultimately, budget seems to affect a film's scale and production quality more than viewers' enjoyment.

In [ ]:
top_n = 25

top_votes_df = exploded_df[['title',
                            'genres_list',
                            'vote_count',
                            'vote_average',
                            'budget',
                            'revenue']].dropna(subset=['genres_list']).copy()

genre_options = sorted(top_votes_df['genres_list'].unique().tolist())

genre_selector = alt.param(
    'selected_genre',
    bind=alt.binding_select(options=genre_options, name='Genre'),
    value='Action'
)

rating_order = [
    'Poor (0-5)',
    'Decent (5-6)',
    'Good (6-7)',
    'Very Good (7-8)',
    'Excellent (8-10)'
]

top_votes_df['rating_band'] = pd.cut(
    top_votes_df['vote_average'], 
    bins=[0, 5, 6, 7, 8, 10],
    labels=rating_order
)

TAB10 = [
    "#1f77b4",  
    "#ff7f0e",  
    "#2ca02c",  
    "#d62728",  
    "#9467bd",  
    "#8c564b",  
    "#e377c2",  
    "#7f7f7f",  
    "#bcbd22",  
    "#17becf",  
]

rating_color_scale = alt.Scale(
    domain=rating_order,
    range=[
        TAB10[0],  
        TAB10[9],  
        TAB10[8],  
        TAB10[2],  
        TAB10[1],  
    ]
)

base = (
    alt.Chart(top_votes_df)
    .transform_window(
        rank='rank(vote_count)',
        sort=[alt.SortField('vote_count', order='descending')],
        groupby=['genres_list']
    )
    .transform_filter(f'datum.rank <= {top_n}')
    .transform_filter('datum.genres_list == selected_genre')
)

bars = base.mark_bar().encode(
    x=alt.X(
        'vote_count:Q',
        title='Number of Votes',
        axis=alt.Axis(format=',d')
    ),
    y=alt.Y(
        'title:N',
        title='Movie',
        sort='-x'  
    ),
    color=alt.Color(
        'rating_band:N',
        title='Rating Band',
        sort=rating_order,
        scale=rating_color_scale
    ),
    tooltip=[
        alt.Tooltip('title:N', title='Movie'),
        alt.Tooltip('genres_list:N', title='Genre'),
        alt.Tooltip('vote_count:Q', title='Total Votes', format=','),
        alt.Tooltip('vote_average:Q', title='Rating', format='.1f'),
        alt.Tooltip('budget:Q', title='Budget', format='$,.0f'),
        alt.Tooltip('revenue:Q', title='Revenue', format='$,.0f')
    ]
).properties(
    width=700,
    height=450,
    title={
        'text': f'Top {top_n} Most Voted Movies by Genre',
        'subtitle': 'Bars show the most-voted movies within the selected genre, colored by rating band'
    }
)

chart = bars.add_params(genre_selector)

# Save to HTML
chart.save('viz_exports/top25_voted_chart.html')

chart


Takeaway: This visualization shows that the most-voted movies in each genre cluster within a narrow rating band, usually between 6 and 8 stars. Vote counts indicate popularity and audience reach but do not guarantee higher ratings. Some genres have top-voted films with consistently strong ratings, while others show more variability. The drop-down filter clarifies how each genre’s most popular titles perform, helping you quickly see which genres reliably produce high-rated movies and which show more fluctuation in audience reception.